# Método TFBGF — Problema Inverso de Condução de Calor (X22)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x22_tfbgf.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Do problema direto ao inverso

No **problema direto** conhecemos o fluxo de calor imposto e calculamos a temperatura resultante. No **problema inverso de condução de calor** (IHCP) faz-se o contrário: a partir de temperaturas medidas, **estima-se o fluxo de calor** que as produziu — situação típica quando a superfície aquecida é inacessível à medição.

Este *notebook* implementa o método **TFBGF** (*Transfer Function Based on Green's Functions*) para o **Problema X22B50T1**: placa com fluxo desconhecido $q(t)$ em $x=0$, superfície isolada em $x=L$ e temperatura inicial $T_0 = 25\ ^\circ$C. A ideia central é tratar a condução como um **sistema linear invariante no tempo**, caracterizado por sua **resposta ao impulso** $h(x,t)$: a temperatura é a convolução $T = q * h$, e estimar $q$ é **desconvoluir** esse produto.

## Estratégia

1. **Problema direto (dados sintéticos).** Um pulso triangular de fluxo conhecido gera temperaturas "medidas" $T(x,t)$ pela solução híbrida do Problema X22.
2. **Resposta impulsiva.** A resposta ao impulso $h(x,t)$ caracteriza o sistema.
3. **Deconvolução via FFT.** No domínio da frequência a convolução vira produto, então $\hat q = \mathcal{F}\{T\}/\mathcal{F}\{h\}$.

**Cuidado numérico:** como a placa é isolada, nem $T$ nem $h$ retornam a zero ao fim da janela, o que causaria vazamento espectral no preenchimento com zeros da FFT. A deconvolução é feita então sobre as **derivadas temporais** dos sinais, $dT/dt = q * (dh/dt)$, que decaem a zero.

Com temperaturas ruidosas, a deconvolução amplifica enormemente o ruído; um **filtro passa-baixa de fase zero** atua como **regularização**, recuperando o fluxo ao custo de suavizar as quinas do pulso.

## Bibliotecas utilizadas

- **NumPy** (`numpy`): computação numérica, a série da solução direta e a **transformada de Fourier** (`numpy.fft`) usada na deconvolução.
- **Matplotlib** (`matplotlib.pyplot`): gráficos.

In [ ]:
import numpy as np                # computação numérica e FFT (numpy.fft)
import matplotlib.pyplot as plt   # gráficos

## Parâmetros e fluxo imposto

Placa de **cobre** ($k = 401$ W/(m·K), $\alpha = 117\times10^{-6}$ m²/s), espessura $L = 0{,}1$ m, discretização $dt = 1$ s até $t_f = 1024$ s. O fluxo desconhecido a ser recuperado é um **pulso triangular**: sobe de $0$ a $10^5$ W/m² em $t = 450$ s e desce a zero em $t = 900$ s.

In [ ]:
k     = 401.0        # condutividade térmica [W/(m·K)]  (cobre)
alpha = 117e-6       # difusividade térmica [m²/s]
L     = 10e-2        # espessura da placa [m]
T0    = 25.0         # temperatura inicial [°C]
dt    = 1.0          # passo de tempo [s]
tf    = 1024.0       # tempo final [s]
M     = 200          # número de termos da série

t = np.arange(0.0, tf, dt)          # instantes de tempo
N = len(t)

# Pulso triangular de fluxo de calor imposto em x=0 (o "desconhecido" a estimar)
qpico, tpico, tbase = 1e5, 450.0, 900.0
q = np.where(t <= tpico, qpico * t / tpico,
             np.where(t <= tbase, qpico * (tbase - t) / (tbase - tpico), 0.0))

mm  = np.arange(1, M + 1)
lam = (mm * np.pi / L)**2 * alpha    # autovalores λ_m = (mπ/L)² α

## Problema direto: temperatura e resposta impulsiva

`temperatura_hibrida(x, q)` avalia a solução do Problema X22 integrando o fluxo discreto de forma exata em cada passo — uma recorrência estável, em vez de uma dupla soma. `resposta_impulsiva(x)` avalia $h(x,t)$. Ambas são calculadas em três posições: $x = 0$, $L/2$ e $L$.

In [ ]:
def temperatura_hibrida(x, q):
    """T(x, t) do problema direto X22, integrando o fluxo discreto (constante por partes)."""
    fator = (1 - np.exp(-lam * dt)) / lam
    decai = np.exp(-lam * dt)
    I = np.zeros((M, N))
    for d in range(1, N):
        I[:, d] = decai * I[:, d-1] + q[d-1] * fator            # recorrência da convolução
    modo0 = (alpha / (k * L)) * dt * np.concatenate(([0.0], np.cumsum(q)[:-1]))  # modo m=0
    cosx  = np.cos(mm * np.pi * x / L)
    return T0 + modo0 + (2 * alpha / (k * L)) * np.einsum('m,mt->t', cosx, I)

def resposta_impulsiva(x):
    """Resposta ao impulso h(x, t) do problema X22 (h em t=0 fixado em 0 por convenção)."""
    cosx = np.cos(mm * np.pi * x / L)
    h = (alpha / (k * L)) * (1 + 2 * np.einsum('m,mt->t', cosx, np.exp(-np.outer(lam, t))))
    h[0] = 0.0
    return h

posicoes = [0.0, L/2, L]
rotulos  = [r'$x=0$', r'$x=L/2$', r'$x=L$']

T_sim = [temperatura_hibrida(x, q) for x in posicoes]
h_pos = [resposta_impulsiva(x) for x in posicoes]
print('Temperaturas e respostas impulsivas calculadas em x = 0, L/2 e L.')

### Temperaturas sintéticas

Estas são as temperaturas que, num experimento, seriam medidas — e a partir das quais o fluxo será estimado. Como a placa é isolada, todas crescem continuamente: o calor injetado não escapa.

In [ ]:
cores    = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00']  # paleta Okabe-Ito
tracados = ['-', '--', '-.', ':', (0, (5, 1))]

fig, ax = plt.subplots()
for Tx, rot, cor, ls in zip(T_sim, rotulos, cores, tracados):
    ax.plot(t, Tx, label=rot, color=cor, linestyle=ls, linewidth=1.5)
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('T(x, t)  (°C)')
ax.set_title('Temperaturas sintéticas — pulso triangular de fluxo de calor')
ax.set_xlim(0, tf)
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Resposta impulsiva

A resposta ao impulso $h(x,t)$ tende ao valor assintótico $\alpha/(kL)$ — o sistema é estável. Em $x=0$ ela parte de um pico (o impulso é aplicado ali); nas demais posições cresce suavemente até o patamar.

In [ ]:
fig, ax = plt.subplots()
for hx, rot, cor, ls in zip(h_pos, rotulos, cores, tracados):
    ax.plot(t[1:], hx[1:], label=rot, color=cor, linestyle=ls, linewidth=1.5)
ax.axhline(alpha/(k*L), color='k', linestyle=':', linewidth=1.0, label=r'$\alpha/(kL)$')
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('h(x, t)  (m²·K/J)')
ax.set_title('Resposta impulsiva do problema X22')
ax.set_xlim(0, 200)
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()
print(f'Valor assintótico α/(kL) = {alpha/(k*L):.3e} m²·K/J')

## Problema inverso: deconvolução

`estima_fluxo` calcula $\hat q = \mathcal{F}\{dT/dt\}\,/\,\mathcal{F}\{dh/dt\}$ e volta ao domínio do tempo pela FFT inversa. O parâmetro opcional `fc` liga um filtro passa-baixa (Butterworth de fase zero) que descarta as altas frequências — desnecessário aqui (dados sem ruído), essencial mais adiante. Sem ruído, o pulso triangular é recuperado quase exatamente nas três posições.

In [ ]:
NR    = 2**20                       # comprimento do zero-padding para a FFT
freqs = np.fft.fftfreq(NR, dt)

def estima_fluxo(Tx, hx, fc=None, ordem=4):
    """Estima q(t) por deconvolução de dT/dt com dh/dt; filtro passa-baixa opcional (fc)."""
    W  = 1.0 if fc is None else 1.0 / (1.0 + (np.abs(freqs)/fc)**(2*ordem))   # Butterworth
    qf = np.fft.fft(np.diff(Tx), NR) / np.fft.fft(np.diff(hx), NR) * W
    return np.real(np.fft.ifft(qf))[:N] / dt

q_est = [estima_fluxo(Tx, hx) for Tx, hx in zip(T_sim, h_pos)]

corte = N - 50     # descarta a borda final da janela (efeito de contorno da FFT)
salto = 12         # espaçamento dos marcadores

fig, ax = plt.subplots()
ax.plot(t[:corte], q[:corte], color='k', linewidth=1.8, label='q imposto')
for qe, rot, cor, mk in zip(q_est, rotulos, cores, ['o', 's', '^']):
    ax.plot(t[:corte:salto], qe[:corte:salto], color=cor, linestyle='none', marker=mk,
            markersize=4, markerfacecolor='none', label=f'q estimado, {rot}')
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('q(t)  (W/m²)')
ax.set_title('Fluxo de calor imposto e estimado por deconvolução')
ax.set_xlim(0, tf)
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Qualidade da estimativa: áreas

A área sob a curva de fluxo (a energia total injetada) é um bom indicador global. As áreas estimadas $A_2$ coincidem com a imposta $A_1$ com diferença muito inferior a $1\%$.

In [ ]:
A1 = np.trapezoid(q[:corte], t[:corte])
print(f'Área do fluxo imposto  A1 = {A1:.4e} W·s/m²\n')
print(f'{"posição":>8} | {"A2 (estimado)":>16} | {"A2 - A1":>12} | {"dif. (%)":>10}')
print('-' * 56)
for qe, rot in zip(q_est, ['x = 0', 'x = L/2', 'x = L']):
    A2 = np.trapezoid(qe[:corte], t[:corte])
    print(f'{rot:>8} | {A2:>16.6e} | {A2-A1:>12.3e} | {100*(A2-A1)/A1:>10.2e}')

### Desvios ponto a ponto

Os maiores desvios ocorrem em $x=0$, como pequenas oscilações no início e no fim da janela — efeito das quinas do pulso triangular e do truncamento temporal. Em $x=L/2$ e $x=L$ os desvios são desprezíveis.

In [ ]:
fig, ax = plt.subplots()
for qe, rot, cor, ls in zip(q_est, rotulos, cores, tracados):
    ax.plot(t[:corte], qe[:corte] - q[:corte], label=rot, color=cor, linestyle=ls, linewidth=1.2)
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('q estimado − q imposto  (W/m²)')
ax.set_title('Desvio entre o fluxo estimado e o imposto')
ax.set_xlim(0, tf)
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Efeito do ruído e o filtro regularizador

Numa medição real há ruído. Adiciona-se às temperaturas em $x=L$ um ruído gaussiano com desvio-padrão $\sigma = 0{,}5\ ^\circ$C. **Sem filtro**, a deconvolução amplifica o ruído em várias ordens de grandeza e a estimativa é inutilizável. **Com** um filtro passa-baixa de $f_c = 0{,}01$ Hz — atuando como regularizador — o pulso é recuperado, ao custo de suavizar suas quinas.

In [ ]:
sigma = 0.5                                  # desvio-padrão do ruído [°C]
rng   = np.random.default_rng(42)            # gerador com semente fixa (reprodutível)
T_ruido = T_sim[2] + rng.normal(0.0, sigma, N)   # temperatura ruidosa em x=L

q_sem_filtro = estima_fluxo(T_ruido, h_pos[2])              # sem regularização
q_com_filtro = estima_fluxo(T_ruido, h_pos[2], fc=0.01)     # filtro fc = 0,01 Hz

fig, (axa, axb) = plt.subplots(1, 2, figsize=(10, 4))
axa.plot(t[:corte], q_sem_filtro[:corte] / 1e8, color=cores[1], linewidth=0.7)
axa.set_xlabel('Tempo (s)')
axa.set_ylabel(r'q estimado  ($\times 10^{8}$ W/m²)')
axa.set_title('Sem filtro')
axa.grid(True)

axb.plot(t[:corte], q[:corte] / 1e5, color='k', linewidth=1.5, label='q imposto')
axb.plot(t[:corte:salto], q_com_filtro[:corte:salto] / 1e5, color=cores[0],
         linestyle='none', marker='o', markersize=4, markerfacecolor='none', label='q estimado')
axb.set_xlabel('Tempo (s)')
axb.set_ylabel(r'q  ($\times 10^{5}$ W/m²)')
axb.set_title('Com filtro ($f_c$ = 0,01 Hz)')
axb.legend()
axb.grid(True)
fig.suptitle(r'Estimativa com temperaturas ruidosas em $x=L$  ($\sigma$ = 0,5 °C)')
plt.tight_layout()
plt.show()

A2r = np.trapezoid(q_com_filtro[:corte], t[:corte])
print(f'Erro de área (com filtro): {100*(A2r-A1)/A1:+.3f} %')

## Verificação: temperatura reconstruída

Como num problema real o fluxo verdadeiro é desconhecido, verifica-se a estimativa **realimentando** o fluxo estimado (do caso ruidoso com filtro) na solução direta e comparando a temperatura resultante em $x=L$ com a medida. O erro médio de poucos décimos de grau confirma que o fluxo estimado descreve corretamente o sistema.

In [ ]:
T_estL = temperatura_hibrida(L, q_com_filtro)    # solução direta alimentada com o fluxo estimado

fig, ax = plt.subplots()
ax.plot(t[:corte], T_sim[2][:corte], color='k', linewidth=1.5, label='T simulada, x=L')
ax.plot(t[:corte:20], T_estL[:corte:20], color=cores[0], linestyle='none', marker='o',
        markersize=4.5, markerfacecolor='none', label='T estimada (via q estimado), x=L')
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('T(L, t)  (°C)')
ax.set_title('Verificação da estimativa: temperatura em x = L')
ax.set_xlim(0, tf)
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

erro_T = np.mean(np.abs(T_estL[5:corte] - T_sim[2][5:corte]))
print(f'Erro médio |T_estimada - T_simulada| em x=L: {erro_T:.4f} °C')